<a href="https://colab.research.google.com/github/AndresAvella09/Proyecto_Multimodal/blob/main/notebooks/12_app_buscador.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/AndresAvella09/Proyecto_Multimodal/blob/main/notebooks/12_app_buscador.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 12 · App web del buscador (Streamlit)

Convierte el buscador de los notebooks 09-11 en una **página web**: se escribe una descripción en español o inglés
y aparecen los pósters y títulos de las películas más parecidas.

| Sección | Qué hace | Resultado |
|---|---|---|
| 1 | Exporta lo que necesita la app: catálogo, miniaturas y vectores (sin entrenar y entrenados) | carpeta `app_buscador/` en Drive |
| 2 | Escribe el código de la app (`app.py`) y sus archivos de configuración | `app.py`, `requirements.txt`, `Dockerfile` |
| 3 | Revisa que la búsqueda dé lo mismo que en el notebook 11 | — |
| 4 | **Lanza la app desde Colab** con un link público temporal | link `https://….trycloudflare.com` |
| 5 | (Opcional) La publica de forma **permanente** en Hugging Face Spaces | link `https://huggingface.co/spaces/…` |

**Requisitos:** notebooks 05, 09, 10 y 11 (usa `posters_mini.parquet`, `e5_txt.npy`, `poster_adapt.npy` y `e5_adapt.npy`).
Corre en CPU, no hace falta GPU.

In [ ]:
import os, sys, json, time, re, subprocess
import numpy as np, pandas as pd

EN_COLAB = 'google.colab' in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    !pip install -q streamlit sentence-transformers

BASE = os.environ.get('PM_BASE', '/content/drive/MyDrive/proyecto_multimodal')
DATA, EMB = f'{BASE}/data', f'{BASE}/embeddings'
if not os.path.exists(f'{EMB}/clip_img.npy') and os.path.exists(f'{EMB}/embeddings/clip_img.npy'):
    EMB = f'{EMB}/embeddings'
APP_DIR = f'{BASE}/app_buscador'            # todo lo que necesita la app queda aquí
os.makedirs(APP_DIR, exist_ok=True)
print('Vectores en:', EMB, '| App en:', APP_DIR)

Mounted at /content/drive
Vectores en: /content/drive/MyDrive/proyecto_multimodal/embeddings | App en: /content/drive/MyDrive/proyecto_multimodal/app_buscador


## 1. Exportar los datos de la app

Solo entran las **películas de cine** (las mismas del catálogo de búsqueda de los notebooks 10 y 11).
Los vectores se guardan en `float16` para que pesen la mitad; la pérdida de precisión no cambia el orden de los resultados.

In [ ]:
def l2(X):
    X = X.astype(np.float32)
    return X / np.linalg.norm(X, axis=1, keepdims=True).clip(min=1e-8)

pelis = pd.read_parquet(f'{DATA}/mmimdb_con_anio.parquet')
mini = pd.read_parquet(f'{DATA}/posters_mini.parquet')
assert (pelis['fila_hdf5'].values == np.arange(len(pelis))).all() and len(mini) == len(pelis)

filas = np.where(pelis['es_pelicula'].fillna(False).values.astype(bool))[0]
cat = pd.DataFrame({
    'fila_hdf5': filas,
    'tconst': pelis['tconst'].values[filas],
    'titulo': pelis['titulo'].fillna('(sin título)').values[filas],
    'anio': pelis['anio'].values[filas],
    'generos': pelis['generos_imdb_hoy'].fillna('').values[filas],
    'jpg': mini['jpg'].values[filas],
})
cat.to_parquet(f'{APP_DIR}/catalogo.parquet', index=False)

vectores = {
    'poster_orig.npy': l2(np.load(f'{EMB}/clip_img.npy')),
    'sinopsis_orig.npy': l2(np.load(f'{EMB}/e5_txt.npy')),
    'poster_adapt.npy': l2(np.load(f'{EMB}/poster_adapt.npy')),
    'sinopsis_adapt.npy': l2(np.load(f'{EMB}/e5_adapt.npy')),
}
for nombre, Vx in vectores.items():
    assert len(Vx) == len(pelis), nombre
    np.save(f'{APP_DIR}/{nombre}', Vx[filas].astype(np.float16))

anios = cat['anio'].dropna().astype(int)
config = {'modelo_poster': 'sentence-transformers/clip-ViT-B-32-multilingual-v1',
          'modelo_sinopsis': 'intfloat/multilingual-e5-base',
          'alfa': 0.4,                      # mejor peso del póster en el notebook 11
          'anio_min': int(anios.min()), 'anio_max': int(anios.max())}
with open(f'{APP_DIR}/config.json', 'w') as fh:
    json.dump(config, fh, indent=2)

print(f'{len(cat):,} películas | años {config["anio_min"]}-{config["anio_max"]}')
for f in sorted(os.listdir(APP_DIR)):
    print(f'  {f:<22} {os.path.getsize(f"{APP_DIR}/{f}") / 1e6:7.1f} MB')

23,547 películas | años 1912-2021
  catalogo.parquet         192.0 MB
  config.json                0.0 MB
  poster_adapt.npy          24.1 MB
  poster_orig.npy           24.1 MB
  sinopsis_adapt.npy        36.2 MB
  sinopsis_orig.npy         36.2 MB


## 2. El código de la app

`app.py` es una app de **Streamlit**. La barra lateral tiene las opciones:

- **Buscar por:** ambos / póster / sinopsis.
- **Peso del póster** (en modo "ambos"): arranca en 0.4, el mejor valor del notebook 11.
- **Representación:** *entrenada* (adaptador del notebook 11) o *sin entrenar*, para mostrar el antes y el después en vivo.
- **Años de estreno** y **número de resultados**.

En modo "ambos", cada resultado dice si lo encontró **más por el póster o más por la sinopsis**: es el desbalance entre
modalidades que estudia el proyecto, visto película por película.

La celda escribe también `requirements.txt`, el tema de colores y un `Dockerfile` para publicarla en Hugging Face (sección 5).

In [ ]:
APP = r'''"""Buscador semántico de películas (Streamlit).

Póster: CLIP ViT-B/32 multilingüe · Sinopsis: multilingual-e5-base.
Los archivos de datos (catalogo.parquet, *.npy, config.json) se buscan en BUSCADOR_DIR
o, si no está definida, en la misma carpeta que este archivo.
"""
import html
import io
import json
import os

import numpy as np
import pandas as pd
import streamlit as st
from PIL import Image

DIR = os.environ.get('BUSCADOR_DIR', os.path.dirname(os.path.abspath(__file__)))

st.set_page_config(page_title='Buscador de películas', page_icon='🎬', layout='wide')

st.markdown("""
<style>
.block-container {padding-top: 2rem; max-width: 1200px;}
.tarjeta-titulo {font-weight: 600; font-size: 0.92rem; line-height: 1.2; margin-top: 0.3rem;}
.tarjeta-meta {font-size: 0.78rem; opacity: 0.75; line-height: 1.25;}
.etiqueta {display: inline-block; font-size: 0.7rem; padding: 1px 7px; border-radius: 10px;
           margin-top: 4px; border: 1px solid rgba(245, 158, 11, 0.6); color: #f59e0b;}
.etiqueta.sinopsis {border-color: rgba(96, 165, 250, 0.6); color: #60a5fa;}
.tarjeta-fin {margin-bottom: 1.4rem;}
</style>
""", unsafe_allow_html=True)


# ---------- datos y modelos (se cargan una sola vez) ----------
@st.cache_resource(show_spinner='Cargando películas...')
def cargar_datos():
    cat = pd.read_parquet(f'{DIR}/catalogo.parquet')
    with open(f'{DIR}/config.json') as fh:
        cfg = json.load(fh)
    cargar = lambda n: np.load(f'{DIR}/{n}').astype(np.float32)
    V = {
        ('poster', 'Entrenada'): cargar('poster_adapt.npy'),
        ('sinopsis', 'Entrenada'): cargar('sinopsis_adapt.npy'),
        ('poster', 'Sin entrenar'): cargar('poster_orig.npy'),
        ('sinopsis', 'Sin entrenar'): cargar('sinopsis_orig.npy'),
    }
    assert all(len(v) == len(cat) for v in V.values())
    return cat, cfg, V


@st.cache_resource(show_spinner='Cargando modelos de lenguaje (la primera vez tarda un poco)...')
def cargar_modelos(nombre_poster, nombre_sinopsis):
    from sentence_transformers import SentenceTransformer
    return SentenceTransformer(nombre_poster), SentenceTransformer(nombre_sinopsis)


cat, CFG, V = cargar_datos()
modelo_clip, modelo_e5 = cargar_modelos(CFG['modelo_poster'], CFG['modelo_sinopsis'])
ANIOS = cat['anio'].fillna(0).astype(int).values


def z(s):
    return (s - s.mean()) / (s.std() + 1e-8)


@st.cache_data(max_entries=200, show_spinner=False)
def codificar(consulta):
    qp = modelo_clip.encode([consulta], convert_to_numpy=True)[0]
    qs = modelo_e5.encode([f'query: {consulta}'], convert_to_numpy=True, normalize_embeddings=True)[0]
    return qp / np.linalg.norm(qp), qs


def buscar(consulta, modo, version, alfa, desde, hasta, k):
    qp, qs = codificar(consulta)
    zp = z(V[('poster', version)] @ qp)
    zs = z(V[('sinopsis', version)] @ qs)
    puntaje = {'Póster': zp, 'Sinopsis': zs}.get(modo, alfa * zp + (1 - alfa) * zs)
    puntaje = np.where((ANIOS >= desde) & (ANIOS <= hasta), puntaje, -np.inf)
    top = [int(i) for i in np.argsort(-puntaje)[:k] if np.isfinite(puntaje[i])]
    por_poster = {i: bool(alfa * zp[i] > (1 - alfa) * zs[i]) for i in top}
    return top, por_poster


# ---------- barra lateral ----------
with st.sidebar:
    st.header('Opciones')
    modo = st.radio('Buscar por', ['Ambos', 'Póster', 'Sinopsis'], horizontal=True,
                    help='Póster: lo que se ve. Sinopsis: lo que se cuenta. Ambos: combina los dos.')
    alfa = st.slider('Peso del póster', 0.0, 1.0, float(CFG['alfa']), 0.1,
                     disabled=(modo != 'Ambos'),
                     help='0 = solo sinopsis · 1 = solo póster. El valor por defecto es el que mejor funcionó.')
    version = st.radio('Representación', ['Entrenada', 'Sin entrenar'], horizontal=True,
                       help='"Entrenada" usa los vectores ajustados con el adaptador del notebook 11.')
    desde, hasta = st.slider('Años de estreno', CFG['anio_min'], CFG['anio_max'],
                             (CFG['anio_min'], CFG['anio_max']))
    k = st.select_slider('Resultados', [6, 12, 18, 24], value=12)
    st.divider()
    st.caption('Póster: CLIP ViT-B/32 multilingüe · Sinopsis: multilingual-e5-base · '
               'Datos: MM-IMDB e IMDb (uso no comercial).')


# ---------- página principal ----------
st.title('🎬 Buscador de películas')
st.write(f'Describe una película **en español o en inglés** y el buscador encuentra las más parecidas '
         f'entre **{len(cat):,}** películas, mirando su **póster**, su **sinopsis** o **ambos**.')

EJEMPLOS = ['una casa embrujada en la nieve', 'a robot in space', 'princesas',
            'un detective investiga un asesinato', 'dibujos animados de colores', 'piratas en el mar']

if 'consulta' not in st.session_state:
    st.session_state.consulta = ''

def usar_ejemplo(texto):
    st.session_state.consulta = texto

consulta = st.text_input('¿Qué película buscas?', key='consulta',
                         placeholder='Ej.: una casa embrujada en la nieve')
st.caption('O prueba con:')
for fila in range(0, len(EJEMPLOS), 3):
    for c, ej in zip(st.columns(3), EJEMPLOS[fila:fila + 3]):
        c.button(ej, on_click=usar_ejemplo, args=(ej,), width='stretch')

consulta = consulta.strip()
if not consulta:
    st.info('Escribe una descripción o elige un ejemplo.')
    st.stop()

with st.spinner('Buscando...'):
    top, por_poster = buscar(consulta, modo, version, alfa, desde, hasta, k)

if not top:
    st.warning(f'No hay películas entre {desde} y {hasta}.')
    st.stop()

resumen = f'**{len(top)} películas** para *"{consulta}"*'
if modo == 'Ambos':
    n_p = sum(por_poster.values())
    resumen += f' · encontradas más por el póster: **{n_p}** · más por la sinopsis: **{len(top) - n_p}**'
st.markdown(resumen)

COLUMNAS = 6
for fila in range(0, len(top), COLUMNAS):
    for c, i in zip(st.columns(COLUMNAS), top[fila:fila + COLUMNAS]):
        r = cat.iloc[i]
        anio = int(r['anio']) if pd.notna(r['anio']) else '?'
        generos = (r['generos'] or '').replace(',', ' · ')
        with c:
            st.image(Image.open(io.BytesIO(r['jpg'])), width='stretch')
            tarjeta = (f'<div class="tarjeta-titulo">{html.escape(str(r["titulo"]))}</div>'
                       f'<div class="tarjeta-meta">{anio} · {html.escape(generos)}</div>')
            if modo == 'Ambos':
                tarjeta += ('<span class="etiqueta">por el póster</span>' if por_poster[i]
                            else '<span class="etiqueta sinopsis">por la sinopsis</span>')
            st.markdown(tarjeta + '<div class="tarjeta-fin"></div>', unsafe_allow_html=True)
'''

with open(f'{APP_DIR}/app.py', 'w') as fh:
    fh.write(APP)

os.makedirs(f'{APP_DIR}/.streamlit', exist_ok=True)
with open(f'{APP_DIR}/.streamlit/config.toml', 'w') as fh:
    fh.write('[theme]\nbase = "dark"\nprimaryColor = "#f59e0b"\n'
             'backgroundColor = "#0f1117"\nsecondaryBackgroundColor = "#1a1d27"\n\n'
             '[server]\nheadless = true\n\n[browser]\ngatherUsageStats = false\n\n'
             '[client]\ntoolbarMode = "minimal"\n')

with open(f'{APP_DIR}/requirements.txt', 'w') as fh:
    fh.write('--extra-index-url https://download.pytorch.org/whl/cpu\n'
             'torch\nsentence-transformers\nstreamlit\npandas\npyarrow\nnumpy\npillow\n')

with open(f'{APP_DIR}/Dockerfile', 'w') as fh:
    fh.write('FROM python:3.11-slim\n'
             'WORKDIR /app\n'
             'COPY requirements.txt .\n'
             'RUN pip install --no-cache-dir -r requirements.txt\n'
             'COPY . .\n'
             'ENV HOME=/tmp HF_HOME=/tmp/hf\n'
             'EXPOSE 8501\n'
             'CMD ["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0"]\n')

print('Archivos de la app:', sorted(os.listdir(APP_DIR)))

Archivos de la app: ['.streamlit', 'Dockerfile', 'app.py', 'catalogo.parquet', 'config.json', 'poster_adapt.npy', 'poster_orig.npy', 'requirements.txt', 'sinopsis_adapt.npy', 'sinopsis_orig.npy']


## 3. Chequeo: ¿la app busca igual que el notebook 11?

Se repite la búsqueda *"una película de terror"* con la misma lógica de la app y se cuenta cuántos de los 12 primeros
resultados son de terror, con la representación sin entrenar y con la entrenada. Con la entrenada debería haber más.

In [ ]:
from sentence_transformers import SentenceTransformer

modelo_clip = SentenceTransformer(config['modelo_poster'])
modelo_e5 = SentenceTransformer(config['modelo_sinopsis'])
V = {n[:-4]: np.load(f'{APP_DIR}/{n}').astype(np.float32) for n in vectores}

def z(s):
    return (s - s.mean()) / (s.std() + 1e-8)

def buscar_prueba(consulta, version, alfa=config['alfa'], k=12):
    qp = modelo_clip.encode([consulta])[0]; qp = qp / np.linalg.norm(qp)
    qs = modelo_e5.encode([f'query: {consulta}'], normalize_embeddings=True)[0]
    s = alfa * z(V[f'poster_{version}'] @ qp) + (1 - alfa) * z(V[f'sinopsis_{version}'] @ qs)
    return cat.iloc[np.argsort(-s)[:k]]

for version in ['orig', 'adapt']:
    r = buscar_prueba('una película de terror', version)
    n = r['generos'].str.contains('Horror').sum()
    print(f"{'sin entrenar' if version == 'orig' else 'entrenada':>12}: {n}/12 de terror | "
          f"{', '.join(r['titulo'].head(4))}")

modules.json:   0%|          | 0.00/341 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.65k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/572 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  539MB            

... (21 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

sin entrenar: 12/12 de terror | The Fear, Midnight Movie, Darkest Night, Popcorn
   entrenada: 12/12 de terror | The Fear, Bloodbath at the House of Death, From Beyond, Evil Dead II


## 4. Lanzar la app desde Colab

Streamlit corre dentro de Colab y **Cloudflare** crea un link público temporal (`https://….trycloudflare.com`)
para abrirla desde cualquier navegador, incluso desde el celular. El link funciona mientras esta sesión de Colab
esté abierta.

La primera búsqueda tarda unos segundos porque la app carga los modelos; después es casi instantánea.

In [ ]:
# Descargar cloudflared (solo la primera vez en la sesión)
if not os.path.exists('/content/cloudflared'):
    !wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /content/cloudflared

# Detener una versión anterior de la app, si la había
for patron in ['streamlit run', 'cloudflared tunnel']:
    subprocess.run(['pkill', '-f', patron])
time.sleep(1)

env = dict(os.environ, BUSCADOR_DIR=APP_DIR)
app_log = open('/content/streamlit.log', 'w')
subprocess.Popen(['streamlit', 'run', f'{APP_DIR}/app.py', '--server.port', '8501', '--server.headless', 'true'],
                 cwd=APP_DIR, env=env, stdout=app_log, stderr=subprocess.STDOUT)

tunel_log = open('/content/cloudflared.log', 'w')
subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://localhost:8501', '--no-autoupdate'],
                 stdout=tunel_log, stderr=subprocess.STDOUT)

url = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/cloudflared.log').read())
    if m:
        url = m.group(0); break

if url:
    print('La app está en:', url)
    print('(si al abrirla aparece un error, espera unos 20 segundos y recarga: la app todavía está arrancando)')
else:
    print('No apareció el link. Revisa los registros:')
    print(open('/content/cloudflared.log').read()[-1500:])
    print(open('/content/streamlit.log').read()[-1500:])

La app está en: https://nice-empty-change-helping.trycloudflare.com
(si al abrirla aparece un error, espera unos 20 segundos y recarga: la app todavía está arrancando)


In [ ]:
# Si algo falla en la app, aquí se ven los errores
print(open('/content/streamlit.log').read()[-3000:])

2026-11-08 04:42:54.556 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.85.254.25:8501




## 5. (Opcional) Publicarla de forma permanente en Hugging Face Spaces

El link de Cloudflare muere cuando se cierra Colab. Para tener un **link fijo** (por ejemplo para la presentación
o para integrarlo en la página final del proyecto) se puede publicar gratis en **Hugging Face Spaces**:

1. Crea una cuenta en [huggingface.co](https://huggingface.co) (gratis).
2. Crea un token con permiso de escritura: *Settings → Access Tokens → Create new token → Write*.
3. En Colab, abre el panel de la **llave** 🔑 (Secretos, a la izquierda), agrega uno llamado `HF_TOKEN` con ese token
   y activa "Acceso al notebook".
4. Cambia `USUARIO_HF` abajo por tu usuario y corre la celda.

La subida tarda unos minutos (~350 MB). Después, Hugging Face construye la app (~5-10 minutos la primera vez);
el avance se ve en la pestaña *Logs* del Space.

In [ ]:
USUARIO_HF = 'tu-usuario'          # <- cambia esto
NOMBRE_SPACE = 'buscador-peliculas'
PUBLICAR = False                   # <- cambia a True para publicar

if PUBLICAR:
    from huggingface_hub import HfApi
    from google.colab import userdata
    api = HfApi(token=userdata.get('HF_TOKEN'))
    repo_id = f'{USUARIO_HF}/{NOMBRE_SPACE}'
    api.create_repo(repo_id, repo_type='space', space_sdk='docker', exist_ok=True)

    readme = ('---\ntitle: Buscador de películas\nemoji: 🎬\ncolorFrom: yellow\ncolorTo: gray\n'
              'sdk: docker\napp_port: 8501\npinned: false\n---\n\n'
              'Buscador semántico de películas (póster + sinopsis) del proyecto '
              '[Proyecto_Multimodal](https://github.com/AndresAvella09/Proyecto_Multimodal).\n')
    with open(f'{APP_DIR}/README.md', 'w') as fh:
        fh.write(readme)

    api.upload_folder(folder_path=APP_DIR, repo_id=repo_id, repo_type='space')
    print('Publicado en: https://huggingface.co/spaces/' + repo_id)
else:
    print('No se publicó (PUBLICAR = False).')

## Resumen

- La carpeta `app_buscador/` en Drive tiene **todo** lo que necesita la app: datos, vectores, código y configuración.
- La app se puede abrir desde Colab (sección 4) o publicar con un link fijo (sección 5).
- Más adelante, la misma app puede crecer con más páginas de Streamlit (mapa de películas, resultados de la comparación temporal)
  para armar la página final del proyecto.